# Quickstart: `pycreditools`

A short on-ramp — not the masterclass. It takes about fifteen cells to go from a
raw applicant base to a credit policy you can read.

The path has two halves, and they are the shape of the actual problem:

1. **A standalone base** — a greenfield book where nothing has rejected anyone yet.
   You meet `CreditPolicy` with nothing else in the way: no incumbent, no swap, no
   masking. One score, one cutoff, three numbers.
2. **The incumbent base** — a book with a legacy policy already in force. This is the
   world most readers actually live in, and it is where the three numbers stop being
   trivial. The quickstart ends here, pointing at the masterclass.

For the full story — swap analysis, rating, P&L, and the calibration problem — see the
masterclass (issue #76).

In [1]:
from pycreditools import (
    CreditPolicy,
    col,
    generate_standalone_sample_data,
    generate_sample_data,
    LEGACY_APPROVAL_QUANTILE,
)

## 1. The standalone base

`generate_standalone_sample_data` returns a greenfield book: no policy has ever run on
it, so **every** applicant has an observed `actual_default`. There is no `approved`
column, no `legacy_score`, no masking — just applicants, a monotone score ladder
(`score_2` … `score_5`, higher is better), and the outcome.

In [2]:
base = generate_standalone_sample_data(n_applicants=40_000, seed=7)

print(f"rows: {len(base):,}")
print(f"actual_default observed for all rows: {base['actual_default'].notna().all()}")
base[["applicant_id", "score_5", "income", "actual_default"]].head()

rows: 40,000
actual_default observed for all rows: True


,applicant_id,score_5,income,actual_default
0,1,853,1803,0.0
1,2,32,1538,1.0
2,3,452,3248,1.0
3,4,164,1502,0.0
4,5,744,5935,0.0


## 2. A minimal `CreditPolicy`

The smallest policy that decides anything: approve everyone whose `score_5` clears a
threshold. The direction is **declared**, not inferred — `"gte"` (keep scores *greater
than or equal to* the cutoff) or `"lte"`. The old shorthand `">="` / `"<="` is rejected;
a policy should say which way it points.

In [3]:
policy = (
    CreditPolicy(
        applicant_id_col="applicant_id",
        score_cols=("score_5",),
        actual_default_col="actual_default",
    )
    .cutoff("Score cutoff", {"score_5": 500}, direction="gte")
)

sim = policy.simulate(base, method="analytical")
policy.describe()

'CreditPolicy:\n  Applicant ID: applicant_id\n  Score columns: score_5\n  Current approval: None\n  Actual default: actual_default\n  Estimated default: None\n  Stages:\n    1. Score cutoff (CutoffStage)\n  Stress Scenarios: None\n  Rating Recipe: None'

## 3. Read the KPIs

A simulation labels every row with two funnel counts: `approved_pre_rate` (survived the
filter and cutoff stages) and `new_approval` (survived the whole funnel, including any
rate stage — i.e. *contracted*). Three metrics come off those two columns, and the whole
point of the next helper is that you can see the denominator each one uses.

In [4]:
def kpis(sim):
    """The ADR 0008 metric contract, read straight off the funnel columns."""
    d = sim.data
    n = len(d)
    approved = d["approved_pre_rate"].sum()      # passed filters + cutoffs (pre take-up)
    contracted = d["new_approval"].sum()         # passed the whole funnel
    weighted_default = (d["simulated_default"] * d["new_approval"]).sum()
    return {
        "approval_rate": approved / n,                # approved / total
        "take_up_rate": contracted / approved,        # contracted / approved
        "default_rate": weighted_default / contracted,  # contracted-weighted
    }


for name, value in kpis(sim).items():
    print(f"{name:>14}: {value:.3f}")

 approval_rate: 0.505
  take_up_rate: 1.000
  default_rate: 0.155


### The three numbers, stated once (ADR 0008)

- **`approval_rate` = approved ÷ total.** Always **pre take-up**. It measures the
  underwriting rule and nothing else.
- **`take_up_rate` = contracted ÷ approved.** A *conversion*, denominated in *approved*.
  On this standalone base it is exactly **1.0** — the policy has no rate stage, so
  everyone approved is counted as contracted.
- **`default_rate`** is **always contracted-weighted**. You cannot default on a loan you
  never took.

There is deliberately **no metric for contracted ÷ total**. It is just
`approval_rate × take_up_rate` — derivable, and naming it invites exactly the confusion
this contract closes. On the standalone base `take_up_rate` is 1.0, so approval and
contracted volume coincide and the distinction looks academic. The next base is where it
stops being academic.

## 4. The incumbent base

`generate_sample_data` returns a book with a legacy policy already in force. Two things
change, and both are real:

- There is now an `approved` column (the incumbent's decision), a `legacy_score`, and a
  `hired` column (who actually contracted).
- `actual_default` is **masked** — it is `NaN` for everyone who did not contract, exactly
  the way production masks it. You only observe repayment on loans you actually made.

In [5]:
incumbent = generate_sample_data(n_applicants=40_000, seed=7)
legacy_cut = float(incumbent["legacy_score"].quantile(LEGACY_APPROVAL_QUANTILE))

masked = incumbent["actual_default"].isna().mean()
print(f"legacy cutoff (legacy_score): {legacy_cut:.0f}")
print(f"actual_default masked (NaN) for {masked:.0%} of rows")
incumbent[["applicant_id", "legacy_score", "score_5", "hired", "actual_default"]].head()

legacy cutoff (legacy_score): 789
actual_default masked (NaN) for 91% of rows


,applicant_id,legacy_score,score_5,hired,actual_default
0,1,817,787,1,0.0
1,2,55,45,0,NaN
2,3,176,228,0,NaN
3,4,314,317,0,NaN
4,5,693,764,0,NaN


## 5. The same policy, now against the incumbent

The reader's real problem is not a blank slate — it is a book with a rule already
running. So the funnel gains the incumbent's own gates and a **rate stage** for take-up:

- **Hard filters** — the bureau knock-outs (`age`, `vl_negativacao`).
- **The legacy cutoff** — on `legacy_score`, at the incumbent's threshold.
- **A `RateStage`** — take-up, read from the observed `hired` outcome
  (`observed_col="hired"`). `calibrate_by="score"` would estimate a rate by score decile
  for anyone without an observed outcome; here every applicant is a keep-in with a real
  `hired` value, so it never fires.

That is why sitting the funnel *inside* the incumbent's approved population matters: it
keeps every contracted applicant's outcome observed, so no rate ever has to be estimated. Approving someone the incumbent
*rejected*, whose outcome you never saw, is the swap-in calibration problem, and that
belongs to the masterclass.

In [6]:
incumbent_policy = (
    CreditPolicy(
        applicant_id_col="applicant_id",
        score_cols=("score_5",),
        actual_default_col="actual_default",
    )
    .filter("Hard filters", (col("age") >= 18) & (col("vl_negativacao") <= 5000))
    .cutoff("Legacy cutoff", {"legacy_score": legacy_cut}, direction="gte")
    .rate("Take-up", base_rate=1.0, observed_col="hired", calibrate_by="score")
)

sim_incumbent = incumbent_policy.simulate(incumbent, method="analytical")

for name, value in kpis(sim_incumbent).items():
    print(f"{name:>14}: {value:.3f}")

 approval_rate: 0.205
  take_up_rate: 0.461
  default_rate: 0.067


## 6. Why the denominators matter

Two of the three numbers just came alive:

- **`take_up_rate` is a real fraction, not 1.0.** Not everyone approved contracts, and the
  rate stage says how many do.
- **`default_rate` moved** — and the honest way to see it is the same-base split just
  below — not the standalone-vs-incumbent numbers, which are different books. It moved for a reason: take-up is **adversely selected on score** — the best-scoring applicants shop around and contract *least*, so the
  contracted book skews toward worse scores than the approved book did.

That is the whole case for ADR 0008 in one measurement. Read default over *approved* and
over *contracted* and they are different numbers — and the contracted one, the one you
actually carry, is the one that matters.

In [7]:
d = sim_incumbent.data
default_over_approved = (d["simulated_default"] * d["approved_pre_rate"]).sum() / d["approved_pre_rate"].sum()
default_over_contracted = (d["simulated_default"] * d["new_approval"]).sum() / d["new_approval"].sum()

print(f"default over approved   (wrong denominator): {default_over_approved:.3f}")
print(f"default over contracted (the ADR 0008 rate): {default_over_contracted:.3f}")

default over approved   (wrong denominator): 0.031
default over contracted (the ADR 0008 rate): 0.067


## 7. Where to go next

You have met `CreditPolicy`, run it on two bases, and read the three numbers that every
surface in the library reports the same way. What the quickstart deliberately left out —
all of it in the **masterclass (issue #76)**:

- **Swap analysis** — who you'd swap in and out against the incumbent, and the P&L of it.
- **The calibration problem** — scoring outcomes for applicants the incumbent rejected,
  where `actual_default` is masked.
- **Rating, sequential cutoffs, hard-filter suggestion, and export.**

The masterclass is the long road. This was the short one.